In [ ]:
import openslide
import matplotlib.pyplot as plt
import numpy as np
import glob
import json
import pandas as pd
from tqdm import tqdm
import matplotlib.backends.backend_pdf
from PIL import Image
import os

In [ ]:
from pyzbar.pyzbar import decode

In [ ]:
from PIL import ImageEnhance

In [ ]:
!ls /nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/


In [ ]:
!ls -lth /nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/2025-03-13 | awk 'NR<=10'

In [ ]:
slides = glob.glob("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/2025-03-13/*.svs")# + glob.glob("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/2025-03-12/*.svs") 
print(f"{len(slides)}")


In [ ]:
results = []
for sfn in tqdm(slides):
    slide = openslide.OpenSlide(sfn)
    dc = decode(slide.associated_images["label"].resize([i//3 for i in slide.associated_images["label"].size]))
    if len(dc):
        qrcd = dc[0].data.decode("utf-8").removesuffix("_mlins")
    else:
        qrcd = "X"
    
        fig, axs = plt.subplots(1,2,  width_ratios=[1, 3])
        axs[0].imshow(slide.associated_images["label"])
        axs[0].set_title(qrcd)
        axs[1].imshow(slide.associated_images["macro"].transpose(Image.ROTATE_270))
        axs[1].set_title(sfn.split("/")[-1])
        #axs[1].set_title(s["Unnamed: 4"])
    slide.close()
    results.append((sfn, qrcd))

In [ ]:
results2 = sorted(results, key=lambda x:x[-1])
res = pd.DataFrame(results2, columns=["path", "a_code"])
#res.iloc[-1]["a_code"] = "A19902"
res

In [ ]:
out_fname = "_20250312"
res.to_csv(f"/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/ocr_results/{out_fname}.csv", index=False)


In [ ]:
out_name = "output_0312.pdf"
if out_name:
    pdf = matplotlib.backends.backend_pdf.PdfPages(out_name)

for i, (sfn, aid) in tqdm(enumerate(results2)):
    slide = openslide.OpenSlide(sfn) #.replace("Slide_Incoming", "Slide_Incoming/svs").replace("2025-01-18", "2025-01-18b"))
    fig, axs = plt.subplots(1,3,  figsize=(12,3), width_ratios=[1, 3,3])
    axs[0].imshow(slide.associated_images["label"].resize([i//3 for i in slide.associated_images["label"].size]))
    axs[0].set_title(aid)
    axs[1].imshow(slide.associated_images["macro"].resize([i//3 for i in slide.associated_images["macro"].size]))
    axs[1].set_title(sfn.split("/")[-1])
    tn = slide.associated_images["thumbnail"]

    #tn = ImageEnhance.Contrast(tn).enhance(8/4)
    #tn = ImageEnhance.Brightness(tn).enhance(3/4)

    axs[2].imshow(tn)
    slide.close()
    
    for ax in axs: ax.axis("off") 
    if out_name: pdf.savefig(fig)

if out_name: pdf.close()

In [ ]:
#slide = openslide.OpenSlide("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/2025-01-25/667241bb-7a1c-1ec7-d1a5-e79988f781f2_201643.svs")
#slide.associated_images["thumbnail"]

In [ ]:
#slide.dimensions


In [ ]:
#slide.read_region((70000,42000), 0, (2048,2048))